# Fink/LSST — Reload & Display : LMC, SMC & Groupe Local

This notebook reads the Parquet files saved by `01_fink_block_lightcurves_lmcsmc.ipynb`  
and reproduces the same visualisations (flux and magnitude) per source group.

**Expected data** in `data_FINK_BLOCK_LC_LMCSMC/`:
- `{group}_fp.parquet`  — forced-photometry light curves
- `{group}_src.parquet` — detection-based light curves (diaSources)
- `flatness_metrics.csv` — pre-computed flatness metrics

No API call is made in this notebook.

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_LMCSMC"
DIR_DATA = f"data_{NB_TAG}"  # input: written by notebook 01
DIR_FIGS = f"figs_{NB_TAG}_02"  # output: figures specific to this notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Display parameters ────────────────────────────────────────────────────────
NC = 20  # maximum number of light curves to plot per group
BANDS = list("ugrizy")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


# ── Plotting mode ────────────────────────────────────────────────────────
# Choisir quels groupes afficher :
#   'all'     -> tous les groupes présents sur disque (comportement par défaut)
#   'calib'   -> seulement les groupes utilisables pour la calibration photométrique
#   'exclude' -> seulement les groupes EXCLUS de la calibration
#                (étoiles variables, objets SSO, transitoires TNS, blazars...)
PLOT_MODE = "all"  # <── changer ici : 'all' | 'calib' | 'exclude'

# Groupes exclus de la calibration (cohérent avec notebook 01).
# Tout groupe dont le nom COMMENCE PAR 'simbad_variable' est aussi exclu.
CALIBRATION_EXCLUDE = {
    "solar_system",
    "tns_transient",
    "gaia_star_variable",
    "vsx_variable",
    "gcvs_variable",
    "spicy_yso",
    "blazar_x3hsp",
    "blazar_4lac",
}

print(f"Data directory   : {os.path.abspath(DIR_DATA)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"Plot mode        : {PLOT_MODE!r}")

## 2. Utility functions (identical to notebook 01)

In [ ]:
def flux_to_mag(flux_nJy, flux_err_nJy=None):
    """Convert PSF flux (nJy) to AB magnitude, propagating uncertainties."""
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def rms_variability(flux):
    """Normalised RMS variability: sigma(<f>) / <f>, computed on finite positive flux values."""
    f = np.asarray(flux, dtype=float)
    f = f[np.isfinite(f) & (f > 0)]
    return float(np.std(f) / np.mean(f)) if len(f) >= 3 else np.nan


print("Utility functions defined.")

## 3. Load Parquet files from disk

In [ ]:
# Auto-discover available groups from file names
fp_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_fp.parquet")))
src_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))


def group_from_path(path, suffix):
    return os.path.basename(path).replace(suffix, "")


groups_fp = {group_from_path(p, "_fp.parquet"): p for p in fp_files}
groups_src = {group_from_path(p, "_src.parquet"): p for p in src_files}
all_groups = sorted(set(groups_fp) | set(groups_src))


# ── Apply PLOT_MODE filter ───────────────────────────────────────────────
def _is_excluded(g):
    return g in CALIBRATION_EXCLUDE or g.startswith("simbad_variable")


if PLOT_MODE == "calib":
    selected_groups = [g for g in all_groups if not _is_excluded(g)]
elif PLOT_MODE == "exclude":
    selected_groups = [g for g in all_groups if _is_excluded(g)]
else:  # 'all'
    selected_groups = list(all_groups)

print(
    f"Groupes sur disque: {len(all_groups)},  sélectionnés (PLOT_MODE={PLOT_MODE!r}): {len(selected_groups)}"
)
for g in all_groups:
    has_fp = "yes" if g in groups_fp else "no"
    has_src = "yes" if g in groups_src else "no"
    sel = "<<" if g in selected_groups else "  "
    label = "[EXCL] " if _is_excluded(g) else "[calib]"
    print(f"  {sel} {label} {g:40s}  fp={has_fp:3s}  src={has_src}")

In [ ]:
# Load all Parquet files and reconstruct the lc_cache dictionary.
# Structure: lc_cache[group][oid] = {'fp': DataFrame, 'src': DataFrame}

lc_cache = {}

for group in all_groups:
    lc_cache[group] = {}

    for tag, path_dict in (("fp", groups_fp), ("src", groups_src)):
        if group not in path_dict:
            continue
        df = pd.read_parquet(path_dict[group])

        # Drop any residual NaN rows in core columns
        df = df.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)

        # Recompute mag / mag_err if absent (e.g. columns were not saved)
        if "mag" not in df.columns or "mag_err" not in df.columns:
            mag, mag_err = flux_to_mag(df["r:psfFlux"].values, df["r:psfFluxErr"].values)
            df["mag"] = mag
            df["mag_err"] = mag_err
            df = df.dropna(subset=["mag", "mag_err"]).reset_index(drop=True)

        # Split by object
        for oid, sub in df.groupby("diaObjectId"):
            oid = str(oid)
            if oid not in lc_cache[group]:
                lc_cache[group][oid] = {"fp": pd.DataFrame(), "src": pd.DataFrame()}
            lc_cache[group][oid][tag] = sub.reset_index(drop=True)

# Summary
print("Objects loaded per group:")
for g in all_groups:
    n_oids = len(lc_cache[g])
    n_pts = sum(len(d["fp"]) + len(d["src"]) for d in lc_cache[g].values())
    print(f"  {g:35s}  {n_oids:3d} objects  {n_pts:6d} data points")

## 4. Load flatness metrics

In [ ]:
csv_path = os.path.join(DIR_DATA, "flatness_metrics.csv")
if os.path.exists(csv_path):
    df_flat = pd.read_csv(csv_path)
    print(f"flatness_metrics.csv loaded: {len(df_flat)} rows")
    print(df_flat.groupby("group")[["n_pts", "rms_var"]].median().round(4))
else:
    df_flat = pd.DataFrame()
    print("flatness_metrics.csv not found — flatness plots will be skipped.")

## 5. Flatness boxplot per group

In [ ]:
if df_flat.empty:
    print("No flatness data available.")
else:
    groups_present = [g for g in all_groups if g in df_flat["group"].unique()]
    bands_present = [b for b in BANDS if b in df_flat["band"].unique()]
    short_labels = [g.replace("_", "\n") for g in groups_present]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(3.2 * len(bands_present), 5), sharey=True)
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = df_flat[df_flat["band"] == band]
        data_per_group = [df_b[df_b["group"] == g]["rms_var"].dropna().values for g in groups_present]
        bp = ax.boxplot(data_per_group, labels=short_labels, patch_artist=True, notch=False, showfliers=True)
        for patch in bp["boxes"]:
            patch.set_facecolor(BAND_COLORS.get(band, "#aaa"))
            patch.set_alpha(0.5)
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.tick_params(axis="x", rotation=60, labelsize=7)
        ax.set_yscale("log")
        ax.axhline(0.05, ls="--", color="green", lw=0.8, alpha=0.6)

    axes[0].set_ylabel("Normalised RMS  σ/<f>")
    fig.suptitle("Flux variability by source class — lower = flatter", fontsize=12, fontweight="bold", y=1.02)
    plt.tight_layout()
    savefig("01_flatness_boxplot_by_group")
    plt.show()

## 6. Light-curve plotting functions

In [ ]:
def rank_oids(group, nc=NC):
    """Return the top-nc object IDs for a group, ranked by total number of data points."""
    scored = [(oid, len(d["fp"]) + len(d["src"])) for oid, d in lc_cache[group].items()]
    return [oid for oid, _ in sorted(scored, key=lambda x: -x[1])[:nc]]


def plot_lc_grid(group, mode="flux", nc=NC):
    """
    Plot a (n_objects × n_bands) grid of light curves for one source group.

    Parameters
    ----------
    group : str   — group name (key of lc_cache)
    mode  : str   — 'flux' (nJy) or 'mag' (AB magnitude)
    nc    : int   — maximum number of objects to plot
    """
    top = rank_oids(group, nc)
    n_obj = len(top)
    if n_obj == 0:
        print(f"  No objects for group {group}.")
        return

    fig, axes = plt.subplots(
        n_obj, len(BANDS), figsize=(2.8 * len(BANDS), 2.6 * n_obj), sharex=False, sharey=False, squeeze=False
    )

    for row_idx, oid in enumerate(top):
        d = lc_cache[group][oid]
        frames = [df for df in (d["fp"], d["src"]) if not df.empty]
        if not frames:
            continue

        df_all = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        # Defensive NaN removal on the concatenated light curve
        df_all = df_all.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(
            drop=True
        )

        for col_idx, band in enumerate(BANDS):
            ax = axes[row_idx][col_idx]
            df_b = df_all[df_all["r:band"] == band].sort_values("r:midpointMjdTai")

            if len(df_b) < 3:
                ax.set_visible(False)
                continue

            # Per-band finite mask
            if mode == "flux":
                mask = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
            else:
                mask = np.isfinite(df_b["mag"].values) & np.isfinite(df_b["mag_err"].values)
            df_b = df_b[mask].reset_index(drop=True)

            if len(df_b) < 3:
                ax.set_visible(False)
                continue

            dt = df_b["r:midpointMjdTai"].values
            dt = dt - dt.min()
            color = BAND_COLORS.get(band, "gray")

            if mode == "flux":
                y, yerr = df_b["r:psfFlux"].values, df_b["r:psfFluxErr"].values
            else:
                y, yerr = df_b["mag"].values, df_b["mag_err"].values
                ax.invert_yaxis()

            ax.errorbar(dt, y, yerr=yerr, fmt="o", ms=3, lw=0.8, elinewidth=0.8, color=color, alpha=0.8)
            rms = rms_variability(df_b["r:psfFlux"].values)
            ax.set_title(f"{band} N={len(df_b)} rms={rms:.3f}", fontsize=7, pad=2, color=color)
            ax.set_xlabel("Δt (days)", fontsize=7)
            ax.tick_params(labelsize=7)

        axes[row_idx][0].set_ylabel(f"{oid}\n{'flux (nJy)' if mode == 'flux' else 'AB mag'}", fontsize=8)

    fig.suptitle(f"Group: {group}  |  mode={mode}", fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    safe = group.replace("/", "_").replace(" ", "_")
    savefig(f"02_lc_{safe}_{mode}")
    plt.show()


print("Plotting functions defined.")

## 7. Courbes de lumière en flux

Contrôlé par `PLOT_MODE` (section 1) : `'all'` · `'calib'` · `'exclude'`.

In [ ]:
# lc_cache est chargé en totalité (tous les groupes).
# groups_to_plot est filtré ici par PLOT_MODE via selected_groups.
groups_to_plot = [g for g in selected_groups if g in lc_cache and len(lc_cache[g]) >= 1]

for group in groups_to_plot:
    n_obj = len(lc_cache[group])
    print(f"\n=== {group} ({n_obj} objects) — flux ===")
    plot_lc_grid(group, mode="flux")

## 8. Courbes de lumière en magnitude

Même sélection `PLOT_MODE` que la section 7.

In [ ]:
for group in groups_to_plot:
    n_obj = len(lc_cache[group])
    print(f"\n=== {group} ({n_obj} objects) — magnitude ===")
    plot_lc_grid(group, mode="mag")

## 9. Detailed view — single object inspector

Set `TARGET_GROUP` and `TARGET_OID` below to inspect any individual object.

In [ ]:
# ── Choose the group and object to inspect ────────────────────────────────────
TARGET_GROUP = all_groups[0]  # ← change here
TOP_OIDS = rank_oids(TARGET_GROUP, nc=5)
TARGET_OID = TOP_OIDS[0]  # ← change here (pick from TOP_OIDS)

print(f"Group    : {TARGET_GROUP}")
print(f"Object   : {TARGET_OID}")
print(f"Top OIDs : {TOP_OIDS}")

# ── Build combined light curve ────────────────────────────────────────────────
d = lc_cache[TARGET_GROUP][TARGET_OID]
frames = [df for df in (d["fp"], d["src"]) if not df.empty]
df_obj = (
    pd.concat(frames, ignore_index=True)
    .drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
    .dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"])
    .sort_values("r:midpointMjdTai")
    .reset_index(drop=True)
)

print(f"\nData points per band:")
print(df_obj.groupby("r:band").size().rename("n_pts").to_frame())

In [ ]:
# ── Detailed flux + magnitude side-by-side plot per band ──────────────────────
bands_obj = [b for b in BANDS if b in df_obj["r:band"].values]

fig, axes = plt.subplots(len(bands_obj), 2, figsize=(12, 3.2 * len(bands_obj)), squeeze=False)

for row_idx, band in enumerate(bands_obj):
    df_b = df_obj[df_obj["r:band"] == band].copy()
    dt = df_b["r:midpointMjdTai"].values
    dt = dt - dt.min()
    color = BAND_COLORS.get(band, "gray")

    # Flux panel
    ax0 = axes[row_idx][0]
    mask_f = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
    ax0.errorbar(
        dt[mask_f],
        df_b["r:psfFlux"].values[mask_f],
        yerr=df_b["r:psfFluxErr"].values[mask_f],
        fmt="o",
        ms=3,
        lw=0.8,
        elinewidth=0.8,
        color=color,
        alpha=0.85,
    )
    rms = rms_variability(df_b["r:psfFlux"].values[mask_f])
    ax0.set_ylabel(f"Flux (nJy) — band {band}", color=color)
    ax0.set_xlabel("Δt (days)")
    ax0.set_title(f"Flux  N={mask_f.sum()}  σ/<f>={rms:.4f}", fontsize=8, color=color)

    # Magnitude panel
    ax1 = axes[row_idx][1]
    if "mag" in df_b.columns and "mag_err" in df_b.columns:
        mask_m = np.isfinite(df_b["mag"].values) & np.isfinite(df_b["mag_err"].values)
        ax1.errorbar(
            dt[mask_m],
            df_b["mag"].values[mask_m],
            yerr=df_b["mag_err"].values[mask_m],
            fmt="o",
            ms=3,
            lw=0.8,
            elinewidth=0.8,
            color=color,
            alpha=0.85,
        )
        ax1.invert_yaxis()
        ax1.set_ylabel(f"AB mag — band {band}", color=color)
        ax1.set_xlabel("Δt (days)")
        ax1.set_title(f"Magnitude  N={mask_m.sum()}", fontsize=8, color=color)
    else:
        ax1.set_visible(False)

fig.suptitle(f"diaObjectId = {TARGET_OID}  |  group = {TARGET_GROUP}", fontsize=11, fontweight="bold", y=1.01)
plt.tight_layout()
savefig(f"03_detail_{TARGET_OID}")
plt.show()

## 10. Calibration summary scatter plot

In [ ]:
if df_flat.empty:
    print("No flatness data available.")
else:
    summary = (
        df_flat.groupby(["group", "band"])
        .agg(
            median_rms=("rms_var", "median"),
            mean_flux=("mean_flux_nJy", "mean"),
            n_obj=("diaObjectId", "nunique"),
            n_pts=("n_pts", "sum"),
        )
        .reset_index()
    )
    bands_present = [b for b in BANDS if b in summary["band"].unique()]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(4.5 * len(bands_present), 5))
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = summary[summary["band"] == band]
        ax.scatter(
            df_b["mean_flux"],
            df_b["median_rms"],
            s=80 * np.sqrt(df_b["n_pts"] / max(df_b["n_pts"].max(), 1) + 0.1),
            c=BAND_COLORS.get(band, "gray"),
            alpha=0.8,
            edgecolors="k",
            linewidths=0.5,
        )
        for _, row in df_b.iterrows():
            ax.annotate(
                row["group"],
                (row["mean_flux"], row["median_rms"]),
                fontsize=6,
                ha="left",
                va="bottom",
                xytext=(3, 2),
                textcoords="offset points",
            )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Mean flux (nJy)")
        ax.set_ylabel("Median σ/<f>")
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.axhline(0.05, ls="--", color="green", lw=1, alpha=0.6, label="5%")
        ax.legend(fontsize=7)

    fig.suptitle(
        "Calibration suitability  (best = bright & flat = bottom-right)",
        fontsize=11,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    savefig("04_calibration_summary")
    plt.show()

    # Final ranking table
    print("\nRanking by photometric flatness (ascending RMS):")
    ranking = (
        df_flat.groupby("group")
        .agg(
            n_objects=("diaObjectId", "nunique"),
            n_points=("n_pts", "sum"),
            median_rms=("rms_var", "median"),
            mean_flux_nJy=("mean_flux_nJy", "mean"),
        )
        .sort_values("median_rms")
        .reset_index()
    )
    ranking["mean_mag"] = -2.5 * np.log10(ranking["mean_flux_nJy"] / AB_FLUX_ZERO)
    display(ranking[["group", "n_objects", "n_points", "median_rms", "mean_mag"]])